# Miyabi baseline (single-thread SwiGLU layer)

Kernel: **Python (Miyabi G)**. Everything below is generated from the
repo by `tools/gen_notebook.py`; edit the repo, not this notebook.

Order: write the sources, build on the login node, then run the sweep on
a compute node with `%%sub`.

Before anything else, make sure the kernel can reach Miyabi (see
pd00_intro.ipynb section 1): if a cell reports "Not logged in to
Miyabi", open a Terminal and run `mount-miyabi`, then restart the
kernel.

In [ ]:
%load_ext miyabi

## 1. Place the sources

One directory per run; safe to re-execute (files are overwritten).

In [ ]:
%%bash
mkdir -p ~/swiglu-mlp-parallel/src ~/swiglu-mlp-parallel/scripts/miyabi ~/swiglu-mlp-parallel/results
ls -d ~/swiglu-mlp-parallel

### `CMakeLists.txt`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(swiglu_mlp_parallel LANGUAGES CXX)

# C++17 is enough (integer std::from_chars), and it is what a bare
# `g++ code/*.cpp` invocation defaults to on Miyabi. Staying there means the
# course submission builds with the same one-liner the README shows.
set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_EXPORT_COMPILE_COMMANDS ON)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE RelWithDebInfo CACHE STRING "" FORCE)
endif()

# Single-threaded baseline only. No OpenMP, no threading library, no external
# dependency of any kind: the course requires code that links nothing beyond
# what the compiler links by itself. No -ffast-math either — the FP64
# correctness check would become meaningless.
add_compile_options(-O3 -Wall -Wextra -Wno-unused-parameter)

if(CMAKE_CXX_COMPILER_ID MATCHES "IntelLLVM")
  add_compile_options(-xHost)
elseif(CMAKE_CXX_COMPILER_ID STREQUAL "GNU")
  add_compile_options(-march=native)
endif()

# One binary, one main(): the submission is compiled as a whole, so the
# correctness check is `swiglu_bench --self-check`, not a second executable.
add_executable(swiglu_bench
  code/swiglu_ref.cpp
  code/swiglu_serial.cpp
  code/self_check.cpp
  code/bench_main.cpp)
target_include_directories(swiglu_bench PRIVATE code)

enable_testing()
add_test(NAME correctness COMMAND swiglu_bench --self-check)

### `code/swiglu.h`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/code/swiglu.h
// SwiGLU feed-forward layer (single layer, inference, FP32).
//
//   gate = W_gate @ x        [d_ff]
//   up   = W_up   @ x        [d_ff]
//   h    = SiLU(gate) * up   [d_ff]
//   out  = W_down @ h        [d_model]
//
// with SiLU(g) = g * sigmoid(g) = g / (1 + exp(-g)).
// x is [tokens][d_model], out is [tokens][d_model].
#pragma once

#include <cstddef>
#include <vector>

namespace swiglu {

// TinyLlama-1.1B feed-forward dimensions (hidden 2048, intermediate 5632).
struct Dims {
  int tokens = 16;
  int d_model = 2048;
  int d_ff = 5632;
};

// Row-major weights, as PyTorch nn.Linear stores them:
//   gate/up: [d_ff][d_model]   down: [d_model][d_ff]
struct Weights {
  std::vector<float> gate;
  std::vector<float> up;
  std::vector<float> down;
};

template <typename T>
struct Matrix {
  int rows = 0;
  int cols = 0;
  std::vector<T> data;

  Matrix() = default;
  Matrix(int r, int c) : rows(r), cols(c), data(std::size_t(r) * c, T{}) {}
  T& at(int r, int c) { return data[std::size_t(r) * cols + c]; }
  const T& at(int r, int c) const { return data[std::size_t(r) * cols + c]; }
  T* row(int r) { return data.data() + std::size_t(r) * cols; }
  const T* row(int r) const { return data.data() + std::size_t(r) * cols; }
};

using MatrixF32 = Matrix<float>;
using MatrixF64 = Matrix<double>;

// Deterministic synthetic data so both developers benchmark identical inputs.
void init_weights(Weights& w, const Dims& d, unsigned seed = 1);
MatrixF32 make_input(const Dims& d, unsigned seed = 2);

// Double-precision, unfused reference (allocates separate gate/up/h buffers).
// Slower on purpose; it exists only to define the expected output.
void swiglu_reference(const MatrixF64& x, const Weights& w, MatrixF64& out);

// Single-threaded FP32 baseline: fused per output channel (README's "fused").
// Every later parallel/optimized version is compared against this: same
// FP accumulation order, so mismatches are visible as bit differences.
void swiglu_serial(const MatrixF32& x, const Weights& w, MatrixF32& out);

// Correctness gate: swiglu_serial vs the FP64 reference on a few shapes.
// Returns the number of failed checks (0 = pass). Lives in self_check.cpp and
// is called by `swiglu_bench --self-check`; the submission must stay a single
// binary, so there is deliberately no second main().
int self_check();

// Effective throughput in GFLOP/s: 3 projections x 2 FLOPs per MAC.
inline double gflops(const Dims& d, double seconds) {
  // 3 projections x 2 FLOPs per MAC x tokens * d_model * d_ff
  const double flops =
      6.0 * d.tokens * d.d_model * d.d_ff;
  return flops / seconds / 1e9;
}

}  // namespace swiglu

### `code/swiglu_ref.cpp`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/code/swiglu_ref.cpp
#include "swiglu.h"

#include <cmath>
#include <cstdint>

namespace swiglu {
namespace {

// Small deterministic PRNG (xorshift32): avoids <random> implementation drift
// between platforms and keeps synthetic inputs reproducible.
struct Rng {
  std::uint32_t s;
  explicit Rng(std::uint32_t seed) : s(seed ? seed : 0x9e3779b9u) {}
  std::uint32_t next() {
    s ^= s << 13;
    s ^= s >> 17;
    s ^= s << 5;
    return s;
  }
  // Uniform in [-1, 1).
  float uniform() {
    return (static_cast<float>(next() >> 8) / 8388608.0f) - 1.0f;
  }
};

// Scale keeps the projections in a range where SiLU neither saturates nor
// vanishes, which is what we want to compare numerics against.
constexpr float kWeightScale = 0.04f;
constexpr float kInputScale = 0.5f;

}  // namespace

void init_weights(Weights& w, const Dims& d, unsigned seed) {
  const std::size_t iface = std::size_t(d.d_ff) * d.d_model;
  const std::size_t down = std::size_t(d.d_model) * d.d_ff;
  w.gate.resize(iface);
  w.up.resize(iface);
  w.down.resize(down);

  Rng rng(seed);
  for (float& v : w.gate) v = rng.uniform() * kWeightScale;
  for (float& v : w.up) v = rng.uniform() * kWeightScale;
  for (float& v : w.down) v = rng.uniform() * kWeightScale;
}

MatrixF32 make_input(const Dims& d, unsigned seed) {
  MatrixF32 x(d.tokens, d.d_model);
  Rng rng(seed);
  for (float& v : x.data) v = rng.uniform() * kInputScale;
  return x;
}

void swiglu_reference(const MatrixF64& x, const Weights& w, MatrixF64& out) {
  const int T = x.rows;
  const int H = x.cols;
  const int I = static_cast<int>(w.gate.size() / std::size_t(H));

  out = MatrixF64(T, H);

  // Deliberately unfused: whole gate array, then whole up array, then h.
  std::vector<double> gate(I), up(I), h(I);
  for (int t = 0; t < T; ++t) {
    for (int i = 0; i < I; ++i) {
      double g = 0.0, u = 0.0;
      const float* wg = w.gate.data() + std::size_t(i) * H;
      const float* wu = w.up.data() + std::size_t(i) * H;
      for (int k = 0; k < H; ++k) {
        const double xk = x.at(t, k);
        g += static_cast<double>(wg[k]) * xk;
        u += static_cast<double>(wu[k]) * xk;
      }
      gate[i] = g;
      up[i] = u;
    }
    for (int i = 0; i < I; ++i) {
      const double g = gate[i];
      h[i] = (g / (1.0 + std::exp(-g))) * up[i];
    }
    for (int r = 0; r < H; ++r) {
      const float* wd = w.down.data() + std::size_t(r) * I;
      double acc = 0.0;
      for (int i = 0; i < I; ++i) {
        acc += static_cast<double>(wd[i]) * h[i];
      }
      out.at(t, r) = acc;
    }
  }
}

}  // namespace swiglu

### `code/swiglu_serial.cpp`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/code/swiglu_serial.cpp
#include "swiglu.h"

#include <cmath>

namespace swiglu {

// Single-threaded FP32 baseline. Fused: SiLU is applied to each gate value the
// moment it is produced, so no [tokens][d_ff] gate/up arrays are materialised.
void swiglu_serial(const MatrixF32& x, const Weights& w, MatrixF32& out) {
  const int T = x.rows;
  const int H = x.cols;
  const int I = static_cast<int>(w.gate.size() / std::size_t(H));

  out = MatrixF32(T, H);
  std::vector<float> h(static_cast<std::size_t>(I));

  for (int t = 0; t < T; ++t) {
    const float* xt = x.row(t);
    for (int i = 0; i < I; ++i) {
      const float* wg = w.gate.data() + std::size_t(i) * H;
      const float* wu = w.up.data() + std::size_t(i) * H;
      float g = 0.f, u = 0.f;
      for (int k = 0; k < H; ++k) {
        const float xk = xt[k];
        g += wg[k] * xk;
        u += wu[k] * xk;
      }
      h[std::size_t(i)] = (g / (1.f + std::exp(-g))) * u;
    }

    float* ot = out.row(t);
    for (int r = 0; r < H; ++r) {
      const float* wd = w.down.data() + std::size_t(r) * I;
      const float* hh = h.data();
      float acc = 0.f;
      for (int i = 0; i < I; ++i) {
        acc += wd[i] * hh[i];
      }
      ot[r] = acc;
    }
  }
}

}  // namespace swiglu

### `code/self_check.cpp`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/code/self_check.cpp
// Correctness gate: FP32 baseline vs FP64 reference.
//
// No main() here on purpose. The course submission is compiled as one program
// from all of code/*.cpp, so the check hangs off `swiglu_bench --self-check`
// instead of being a second executable.
#include "swiglu.h"

#include <algorithm>
#include <cmath>
#include <cstdio>
#include <vector>

namespace {

struct Stats {
  double max_abs = 0.0;
  double max_rel = 0.0;  // max|a-b| / max(1, |ref|)
  double mean_abs = 0.0;
};

Stats compare(const swiglu::MatrixF32& got, const swiglu::MatrixF64& ref) {
  Stats s;
  double sum = 0.0;
  const std::size_t n = ref.data.size();
  for (std::size_t i = 0; i < n; ++i) {
    const double a = static_cast<double>(got.data[i]);
    const double b = ref.data[i];
    const double d = std::fabs(a - b);
    s.max_abs = std::max(s.max_abs, d);
    s.max_rel = std::max(s.max_rel, d / std::max(1.0, std::fabs(b)));
    sum += d;
  }
  s.mean_abs = sum / static_cast<double>(n);
  return s;
}

}  // namespace

namespace swiglu {

int self_check() {
  // Tolerance for FP32 accumulation over d_model / d_ff terms against an FP64
  // reference; measured values sit ~3 orders of magnitude below this.
  constexpr double kMaxAbsTol = 2e-3;
  constexpr double kMaxRelTol = 1e-5;

  int failures = 0;
  const Dims dims[] = {
      {1, 2048, 5632},   // single token (decode)
      {16, 2048, 5632},  // default
      {128, 2048, 5632}, // batch case
      {4, 256, 512},     // small, fast sanity case
  };

  for (const auto& d : dims) {
    std::printf("[tokens=%d d_model=%d d_ff=%d]\n", d.tokens, d.d_model,
                d.d_ff);

    Weights w;
    init_weights(w, d);
    MatrixF32 x = make_input(d);

    MatrixF64 xd(d.tokens, d.d_model);
    for (std::size_t i = 0; i < x.data.size(); ++i) {
      xd.data[i] = static_cast<double>(x.data[i]);
    }

    MatrixF64 ref;
    swiglu_reference(xd, w, ref);

    MatrixF32 serial;
    swiglu_serial(x, w, serial);

    const Stats ss = compare(serial, ref);
    std::printf("  serial vs fp64 ref: max_abs=%.3e max_rel=%.3e mean_abs=%.3e\n",
                ss.max_abs, ss.max_rel, ss.mean_abs);

    if (ss.max_abs > kMaxAbsTol || ss.max_rel > kMaxRelTol) {
      std::printf("  FAIL: serial out of tolerance\n");
      ++failures;
    }
    std::printf("\n");
  }

  std::printf("%s\n", failures == 0 ? "ALL CHECKS PASSED" : "CHECKS FAILED");
  return failures;
}

}  // namespace swiglu

### `code/bench_main.cpp`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/code/bench_main.cpp
// Benchmark harness: median wall-clock latency + GFLOPS, CSV on stdout.
//
//   ./swiglu_bench --tokens 16 --warmup 2 --repeat 20
//   ./swiglu_bench --self-check                 # correctness gate
//   ./swiglu_bench --tokens 4 --dump out.txt    # output for external checks
//
// Single-threaded baselines only; sweeps are driven by
// scripts/run_baseline.sh.

#include "swiglu.h"

#include <algorithm>
#include <charconv>
#include <chrono>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <string>
#include <vector>

#include <unistd.h>

namespace {

struct Args {
  std::string impl = "serial";  // serial | reference
  swiglu::Dims dims{};
  int warmup = 2;
  int repeat = 20;
  bool header = true;
  unsigned seed = 1;
  bool self_check = false;
  std::string dump;  // dump the output matrix here (cross-check vs numpy)
};

void usage() {
  std::printf(
      "usage: swiglu_bench [--impl serial|reference] [--tokens T]\n"
      "                    [--d-model H] [--d-ff I] [--warmup N] [--repeat N]\n"
      "                    [--seed N] [--no-header] [--self-check]\n"
      "                    [--dump FILE]\n");
}

int to_int(const char* s) {
  int v = 0;
  auto [p, ec] = std::from_chars(s, s + std::strlen(s), v);
  if (ec != std::errc()) {
    std::fprintf(stderr, "invalid integer: %s\n", s);
    std::exit(2);
  }
  return v;
}

Args parse(int argc, char** argv) {
  Args a;
  for (int i = 1; i < argc; ++i) {
    const std::string k = argv[i];
    auto next = [&]() -> const char* {
      if (i + 1 >= argc) {
        std::fprintf(stderr, "missing value for %s\n", k.c_str());
        std::exit(2);
      }
      return argv[++i];
    };
    if (k == "--impl") a.impl = next();
    else if (k == "--tokens") a.dims.tokens = to_int(next());
    else if (k == "--d-model") a.dims.d_model = to_int(next());
    else if (k == "--d-ff") a.dims.d_ff = to_int(next());
    else if (k == "--warmup") a.warmup = to_int(next());
    else if (k == "--repeat") a.repeat = to_int(next());
    else if (k == "--seed") a.seed = static_cast<unsigned>(to_int(next()));
    else if (k == "--no-header") a.header = false;
    else if (k == "--self-check") a.self_check = true;
    else if (k == "--dump") a.dump = next();
    else if (k == "-h" || k == "--help") { usage(); std::exit(0); }
    else { std::fprintf(stderr, "unknown flag: %s\n", k.c_str()); std::exit(2); }
  }
  return a;
}

std::string compiler_id() {
#if defined(__NVCOMPILER)
  return "nvc++";
#elif defined(__INTEL_LLVM_COMPILER) || defined(__INTEL_COMPILER)
  return "icpx";
#elif defined(__clang__)
  return "clang++";
#elif defined(__GNUC__)
  return "g++";
#else
  return "unknown";
#endif
}

std::string host_name() {
  char buf[256] = {0};
  if (gethostname(buf, sizeof(buf) - 1) != 0) return "unknown";
  std::string s(buf);
  const auto dot = s.find('.');
  return dot == std::string::npos ? s : s.substr(0, dot);
}

double median(std::vector<double> v) {
  std::sort(v.begin(), v.end());
  const std::size_t n = v.size();
  if (n % 2 == 1) return v[n / 2];
  return 0.5 * (v[n / 2 - 1] + v[n / 2]);
}

}  // namespace

int main(int argc, char** argv) {
  const Args a = parse(argc, argv);

  if (a.self_check) return swiglu::self_check() == 0 ? 0 : 1;

  swiglu::Weights w;
  swiglu::init_weights(w, a.dims, a.seed);
  swiglu::MatrixF32 x = swiglu::make_input(a.dims, a.seed + 1);
  swiglu::MatrixF32 out;
  swiglu::MatrixF64 xd, outd;

  const bool reference = a.impl == "reference";
  if (reference) {
    xd = swiglu::MatrixF64(a.dims.tokens, a.dims.d_model);
    for (std::size_t i = 0; i < x.data.size(); ++i) {
      xd.data[i] = static_cast<double>(x.data[i]);
    }
  }

  auto run_once = [&]() {
    if (reference) {
      swiglu::swiglu_reference(xd, w, outd);
    } else if (a.impl == "serial") {
      swiglu::swiglu_serial(x, w, out);
    } else {
      std::fprintf(stderr, "unknown impl: %s\n", a.impl.c_str());
      std::exit(2);
    }
  };

  run_once();  // also warms the page cache for the weight arrays
  for (int i = 0; i < a.warmup; ++i) run_once();

  std::vector<double> samples;
  samples.reserve(static_cast<std::size_t>(a.repeat));
  for (int i = 0; i < a.repeat; ++i) {
    const auto t0 = std::chrono::steady_clock::now();
    run_once();
    const auto t1 = std::chrono::steady_clock::now();
    samples.push_back(std::chrono::duration<double>(t1 - t0).count());
  }

  const double med = median(samples);
  const double best = *std::min_element(samples.begin(), samples.end());

  // Verifiability: a checksum lets any two runs (impl, thread count, compiler,
  // node) be compared without shipping matrices around, and --dump writes the
  // raw output so it can be diffed against an independent implementation.
  const swiglu::MatrixF32& y = out;
  double checksum = 0.0;
  double abs_sum = 0.0;
  if (!reference) {
    for (const float v : y.data) {
      checksum += static_cast<double>(v);
      abs_sum += std::fabs(static_cast<double>(v));
    }
  }

  if (!a.dump.empty() && !reference) {
    std::FILE* f = std::fopen(a.dump.c_str(), "w");
    if (!f) {
      std::fprintf(stderr, "cannot write %s\n", a.dump.c_str());
      return 2;
    }
    std::fprintf(f, "%d %d\n", y.rows, y.cols);
    for (int r = 0; r < y.rows; ++r) {
      for (int c = 0; c < y.cols; ++c) {
        std::fprintf(f, "%s%.9g", c ? "," : "", static_cast<double>(y.at(r, c)));
      }
      std::fputc('\n', f);
    }
    std::fclose(f);
  }

  if (a.header) {
    std::printf(
        "impl,host,compiler,tokens,d_model,d_ff,repeat,median_ms,min_ms,"
        "gflops,checksum,abs_sum\n");
  }
  std::printf("%s,%s,%s,%d,%d,%d,%d,%.4f,%.4f,%.2f,%.9g,%.9g\n",
              a.impl.c_str(), host_name().c_str(), compiler_id().c_str(),
              a.dims.tokens, a.dims.d_model, a.dims.d_ff, a.repeat, med * 1e3,
              best * 1e3, swiglu::gflops(a.dims, med), checksum, abs_sum);
  return 0;
}

### `code/problem.md`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/code/problem.md
# SwiGLU feed-forward layer of a transformer LLM (single layer, inference)

## 1. The problem

A transformer LLM (GPT/Llama/Qwen family) is a stack of identical blocks, and
each block ends with a *feed-forward network* (FFN) that does most of the
floating-point work. In Llama-style models that FFN is called **SwiGLU** and
consists of three matrix-vector products plus a nonlinearity:

```
gate = W_gate · x        # [d_ff]
up   = W_up   · x        # [d_ff]
h    = SiLU(gate) ⊙ up   # elementwise, SiLU(g) = g / (1 + exp(-g))
out  = W_down · h        # [d_model]
```

where `x` is one token's activation vector of length `d_model`, and the weights
are `W_gate, W_up ∈ R^{d_ff × d_model}`, `W_down ∈ R^{d_model × d_ff}`. For the
Llama-2/TinyLlama shape used here, `d_model = 2048`, `d_ff = 5632`, so one layer
holds 3 · 2048 · 5632 = 34.6 M weights (138 MB in FP32).

Inference is batched: `T` tokens are pushed through the layer at once, which is
just the same three matrix products with an `T × d_model` input. Nothing here is
approximate: the output must match a straightforward reference computation.

Why this is a reasonable parallel target:

- The three projections are independent output channels per token: with
  `d_ff = 5632` and 72+ cores there is abundant parallelism (issue #6).
- The arithmetic is simple enough to be checked exactly (see §3), so speed can
  be studied without arguing about correctness.
- It is genuinely memory-heavy: for `T = 16` the layer touches 2.2 GB of weights
  (138 MB read 16 times) but does only 1.107 GFLOP, so a naive parallel version
  is limited by memory bandwidth, not by FLOPs. Measuring *where* the time goes
  is therefore part of the exercise, not a side note.

This baseline deliberately does **one layer, one forward pass, synthetic
weights**. Training a real LLM is out of scope; extracting the real weights of
one TinyLlama layer is an optional extra, not part of the submitted code.

## 2. The baseline code (how it solves the problem)

Files (all C++, no external libraries — only the standard library that the
compiler links by itself):

| file | content |
|---|---|
| `code/swiglu.h` | shapes (`Dims`), weight/activation containers, FP32/FP64 matrix type, `gflops()` |
| `code/swiglu_ref.cpp` | deterministic synthetic data + FP64 unfused reference implementation |
| `code/swiglu_serial.cpp` | the baseline: single-threaded FP32, fused |
| `code/self_check.cpp` | correctness gate, `swiglu::self_check()` |
| `code/bench_main.cpp` | the `main()`: timing, CSV output, `--self-check`, `--dump` |

Build (one command, no build system required):

```sh
g++ -O3 -std=c++17 code/*.cpp -o swiglu_bench
```

Run:

```sh
./swiglu_bench --tokens 16 --repeat 20      # one CSV row: latency + GFLOP/s
./swiglu_bench --self-check                 # exit 0 iff within tolerance
./swiglu_bench --tokens 4 --dump out.txt    # output matrix for external checks
```

Everything is configurable from the command line: `--tokens`, `--d-model`,
`--d-ff`, `--repeat`, `--warmup`, `--seed`. The default (`tokens=16`,
`repeat=20`) finishes in a few seconds; the largest sweep value (128 tokens)
takes about 1.4 s per run on one core, which is why sweeps are driven by
`scripts/run_baseline.sh`.

Implementation notes:

- **Fused, not materialised.** Gate and up are accumulated together, and SiLU
  is applied immediately, so no `T × d_ff` gate/up arrays are ever written to
  memory. Only the `d_ff` vector `h` exists per token.
- **Row-major weights**, stored exactly as PyTorch's `nn.Linear` does
  (`W[r][c]` for output channel `r`), so real weights can be dropped in later
  without a transposition.
- **Deterministic synthetic input** (xorshift32, fixed seed) so two people
  benchmarking get the same numbers; weights are scaled to 0.04 and inputs to
  0.5 so SiLU neither saturates nor vanishes and FP32 error stays meaningful.
- **No `-ffast-math`.** Reordering the sums would make the FP32-vs-FP64
  comparison meaningless, and the baseline is the reference every later version
  is judged against.

## 3. How to verify the result

1. **Self-check (in the submitted code).** `code/self_check.cpp` recomputes the
   same layer in FP64 with a deliberately naive, unfused formulation (separate
   `gate`, `up`, `h` buffers) and compares against the FP32 baseline for
   `tokens = 1, 16, 128, 4`. It fails if `max_abs > 2e-3` or
   `max_rel > 1e-5`; measured values are ~1e-7, i.e. four orders of magnitude
   inside the tolerance (FP32 accumulation error over 2048 and 5632 terms).

   ```
   $ ./swiglu_bench --self-check
   [tokens=16 d_model=2048 d_ff=5632]
     serial vs fp64 ref: max_abs=8.638e-07 max_rel=8.638e-07 mean_abs=1.102e-07
   ALL CHECKS PASSED
   ```

2. **Checksum column.** Every CSV row carries `checksum` and `abs_sum` of the
   output matrix. Any two runs (different compiler, node, thread count later)
   must agree to ~1e-6 relative; that catches "fast but wrong" variants without
   shipping matrices around.

3. **External cross-check.** `--dump out.txt` writes `rows cols` followed by the
   matrix as comma-separated values. Recomputing that in Python (`tokens`,
   `d_model`, `d_ff` are in the CSV row; the same xorshift32 weights can be
   regenerated in a dozen lines) gives an independent check of the
   implementation, not just of its internal consistency.

## 4. What is *not* done yet

This is the baseline only. Multicore parallelisation, thread scaling, SIMD and
blocking are the next steps, tracked as issues #6–#8 in `docs/ISSUES.md`.

### `scripts/run_baseline.sh`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/scripts/run_baseline.sh
#!/bin/bash
# Baseline sweep: one process per token count, single thread, CSV to stdout.
#
#   scripts/run_baseline.sh > results/baseline.csv
#   TOKENS="1 16" scripts/run_baseline.sh        # shorter sweep
#
# Everything here is single-threaded on purpose: this file records the number
# that future parallel versions are scored against.
set -euo pipefail

BUILD=${BUILD:-build}
BENCH=${BENCH:-$BUILD/swiglu_bench}
TOKENS=${TOKENS:-"1 2 4 8 16 32 64 128"}
REPEAT=${REPEAT:-20}

if [[ ! -x "$BENCH" ]]; then
  echo "missing $BENCH (configure with cmake -B $BUILD -S . && cmake --build $BUILD)" >&2
  exit 1
fi

# Keep BLAS/OpenMP-linked runtimes from spawning helpers behind our back.
export OMP_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export MKL_NUM_THREADS=1

first=1
for t in $TOKENS; do
  if [[ $first == 1 ]]; then
    "$BENCH" --impl serial --tokens "$t" --repeat "$REPEAT"
    first=0
  else
    "$BENCH" --impl serial --tokens "$t" --repeat "$REPEAT" --no-header
  fi
done

### `tools/xcheck.py`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/tools/xcheck.py
#!/usr/bin/env python3
"""Independent cross-check of `swiglu_bench --dump` without numpy.

Re-implements the layer in plain Python doubles, with the synthetic inputs
rounded to FP32 exactly as the C++ side does, and compares against a dumped
matrix. Agreement to ~1e-8 absolute means the C++ code is not just
self-consistent but actually computes the SwiGLU layer.

    ./swiglu_bench --tokens 4 --d-model 8 --d-ff 8 --dump /tmp/out.txt --repeat 2
    python3 tools/xcheck.py /tmp/out.txt --tokens 4 --d-model 8 --d-ff 8
"""
from __future__ import annotations

import argparse
import math
import struct


def f32(v: float) -> float:
    """Round a Python float to IEEE binary32, like the C++ side does."""
    return struct.unpack("<f", struct.pack("<f", v))[0]


class Rng:
    """xorshift32, identical to Rng in code/swiglu_ref.cpp."""

    def __init__(self, seed: int) -> None:
        self.s = seed if seed else 0x9E3779B9

    def next(self) -> int:
        s = self.s
        s ^= (s << 13) & 0xFFFFFFFF
        s ^= s >> 17
        s ^= (s << 5) & 0xFFFFFFFF
        self.s = s & 0xFFFFFFFF
        return self.s

    def uniform(self) -> float:
        # (float(n>>8) / 8388608.0f) - 1.0f, all in binary32.
        return f32(f32((self.next() >> 8) / 8388608.0) - 1.0)


def weights(tokens: int, H: int, I: int, seed: int) -> tuple[list[float], ...]:
    rng = Rng(seed)
    gate = [f32(rng.uniform() * 0.04) for _ in range(I * H)]
    up = [f32(rng.uniform() * 0.04) for _ in range(I * H)]
    down = [f32(rng.uniform() * 0.04) for _ in range(H * I)]
    return gate, up, down


def forward(T: int, H: int, I: int, seed: int) -> list[list[float]]:
    gate_w, up_w, down_w = weights(T, H, I, seed)
    rng = Rng(seed + 1)
    x = [[f32(rng.uniform() * 0.5) for _ in range(H)] for _ in range(T)]

    out = []
    for t in range(T):
        h = []
        for i in range(I):
            g = 0.0
            u = 0.0
            for k in range(H):
                xk = x[t][k]
                g += gate_w[i * H + k] * xk
                u += up_w[i * H + k] * xk
            h.append((g / (1.0 + math.exp(-g))) * u)
        row = []
        for r in range(H):
            acc = 0.0
            for i in range(I):
                acc += down_w[r * I + i] * h[i]
            row.append(acc)
        out.append(row)
    return out


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("dump")
    ap.add_argument("--tokens", type=int, default=4)
    ap.add_argument("--d-model", type=int, default=8)
    ap.add_argument("--d-ff", type=int, default=8)
    ap.add_argument("--seed", type=int, default=1)
    a = ap.parse_args()

    with open(a.dump) as f:
        rows, cols = (int(v) for v in f.readline().split())
        got = [[float(v) for v in line.split(",")] for line in f if line.strip()]

    if (rows, cols) != (a.tokens, a.d_model) or len(got) != rows:
        print(f"shape mismatch: header {rows}x{cols}, rows {len(got)}")
        return 1

    want = forward(a.tokens, a.d_model, a.d_ff, a.seed)
    worst = max(abs(got[r][c] - want[r][c])
                for r in range(rows) for c in range(cols))
    scale = max(1.0, max(abs(v) for row in want for v in row))
    print(f"max |diff| = {worst:.3e}  (relative to max |value| {scale:.3e}: "
          f"{worst / scale:.3e})")
    return 0 if worst / scale < 1e-6 else 1


if __name__ == "__main__":
    raise SystemExit(main())

### `scripts/miyabi/env_g.sh`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/scripts/miyabi/env_g.sh
#!/bin/bash
# Miyabi-G environment (aarch64): NVIDIA GH200 — Grace CPU 72c @3.0 GHz,
# LPDDR5X 120 GB at 512 GB/s — plus an H100. Login host: miyabi-g.jcahpc.jp
#
# NOTE: this project times the CPU path only. Grace (512 GB/s) and Miyabi-C
# HBM2e (3.2 TB/s) are different memory regimes, so -c and -g numbers must be
# reported separately and never mixed in one scaling table.
module purge
module load nvidia/25.3 nv-hpcx/25.3

export CC=nvc
export CXX=nvc++

### `scripts/miyabi/env_c.sh`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/scripts/miyabi/env_c.sh
#!/bin/bash
# Miyabi-C environment (x86_64): 2x Intel Xeon Max 9480, 112 cores,
# 128 GiB HBM2e at 3.2 TB/s. Login host: miyabi-c.jcahpc.jp
#
# Sourced by scripts/miyabi/*.pbs and usable interactively on the login node.
module purge
module load intel/2025.0.0

export CC=icx
export CXX=icpx

### `scripts/miyabi/run_baseline_g.pbs`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/scripts/miyabi/run_baseline_g.pbs
#!/bin/bash
#PBS -q lecture1-g
#PBS -l select=1:mpiprocs=1
#PBS -l walltime=00:14:00
#PBS -W group_list=gt81
#PBS -j oe

# Single-threaded SwiGLU baseline on one Miyabi-G (Grace) core.
#
#   cd <your checkout> && qsub scripts/miyabi/run_baseline_g.pbs
#
# group_list=gt81 is fixed by the course ("you do not need to and should never
# change it", pd00_intro.ipynb). Queues are lecture1-{mig,g,c} during the
# lecture slot (Mon 16:50-18:35) and lecture-{mig,g,c} outside it; the 15 minute
# educational walltime cap is why this asks for 14:00.

cd "$PBS_O_WORKDIR" || exit 1
source "$PBS_O_WORKDIR/scripts/miyabi/env_g.sh"

exec 2>&1
set -x

mkdir -p results
echo "=== host/compiler ==="
hostname
"$CXX" --version | head -2

cmake -B build -S . -DCMAKE_BUILD_TYPE=RelWithDebInfo || exit 1
cmake --build build -j || exit 1
ctest --test-dir build --output-on-failure || exit 1

REPEAT=${REPEAT:-20} scripts/run_baseline.sh | tee "results/baseline_$(hostname -s).csv"

### `scripts/miyabi/run_baseline_c.pbs`

In [ ]:
%%writefile ~/swiglu-mlp-parallel/scripts/miyabi/run_baseline_c.pbs
#!/bin/bash
#PBS -q lecture1-c
#PBS -l select=1:mpiprocs=1
#PBS -l walltime=00:14:00
#PBS -W group_list=gt81
#PBS -j oe

# Single-threaded SwiGLU baseline on one Miyabi-C (Xeon Max) core.
#
#   cd <your checkout> && qsub scripts/miyabi/run_baseline_c.pbs
#
# group_list=gt81 is fixed by the course ("you do not need to and should never
# change it", pd00_intro.ipynb). Queues are lecture1-{mig,g,c} during the
# lecture slot (Mon 16:50-18:35) and lecture-{mig,g,c} outside it; the 15 minute
# educational walltime cap is why this asks for 14:00.

cd "$PBS_O_WORKDIR" || exit 1
source "$PBS_O_WORKDIR/scripts/miyabi/env_c.sh"

exec 2>&1
set -x

mkdir -p results
echo "=== host/compiler ==="
hostname
"$CXX" --version | head -2

cmake -B build -S . -DCMAKE_BUILD_TYPE=RelWithDebInfo || exit 1
cmake --build build -j || exit 1
ctest --test-dir build --output-on-failure || exit 1

REPEAT=${REPEAT:-20} scripts/run_baseline.sh | tee "results/baseline_$(hostname -s).csv"

## 2. Build and test on the login node

Small and short enough to run on the login node. The build is the
assignment's one-liner (no external library, one `main()`), and
`--self-check` is the gate: the FP32 baseline must match the FP64
reference to 2e-3 / 1e-5.

In [ ]:
%%bash
set -e
cd ~/swiglu-mlp-parallel
chmod +x scripts/run_baseline.sh
mkdir -p build
g++ -O3 -std=c++17 -Wall code/*.cpp -o build/swiglu_bench
./build/swiglu_bench --self-check

## 3. Single-thread baseline on a compute node

`%%sub` submits this cell as a PBS job, follows it until it exits, and
streams stdout/stderr. Change `lecture1-g` to `lecture-g` outside the
lecture slot (Mon 16:50-18:35). `group_list=gt81` is fixed by the
course. The walltime cap for coursework is 15 minutes, hence 14:00.

`TOKENS` is deliberately short here: 128 tokens x 20 repeats at
~1.4 s/run already eats a few minutes on one core.

In [ ]:
%%sub
#PBS -q lecture1-g
#PBS -l walltime=00:14:00
set -e
cd ~/swiglu-mlp-parallel
echo "host=$(hostname) user=$(id -un) pwd=$(pwd)"
nproc; cc --version | head -2
TOKENS="1 16 128" REPEAT=5 scripts/run_baseline.sh | tee results/baseline_$(hostname -s).csv

## 4. Cross-check one matrix outside the C++ code

`--dump` writes the output matrix; `tools/xcheck.py` recomputes it in
plain Python (no numpy) from the same xorshift32 seeds. This is the
check that the C++ code is correct, not merely self-consistent.

In [ ]:
%%bash
set -e
cd ~/swiglu-mlp-parallel
./build/swiglu_bench --tokens 4 --d-model 8 --d-ff 8 --repeat 2 --dump /tmp/out.txt
python3 tools/xcheck.py /tmp/out.txt --tokens 4 --d-model 8 --d-ff 8

## 5. Copy `code/` into the assignment directory

The submission takes **all and only files under `code/`**, so before
submitting, copy the five sources plus `problem.md` into the `code/`
directory that sits next to pd02_app (no notebooks, no results, no
`scripts/`):

```sh
cp ~/swiglu-mlp-parallel/code/* <assignment-dir>/code/
```

`code/` must contain exactly: `swiglu.h`, `swiglu_ref.cpp`,
`swiglu_serial.cpp`, `self_check.cpp`, `bench_main.cpp`, `problem.md`.

## 6. Reading the result

One CSV row per token count:

```
impl,host,compiler,tokens,d_model,d_ff,repeat,median_ms,min_ms,gflops,checksum,abs_sum
```

`checksum`/`abs_sum` let you compare a compute-node run with a Mac run
without shipping matrices.

Keep `host` and `compiler` — Grace (512 GB/s) and Xeon Max (3.2 TB/s)
numbers must never be mixed in one table. Next steps (parallel version,
thread sweep) are issues #6-#8 in `docs/ISSUES.md`; by then the
`OMP_NUM_THREADS` / `select=1:mpiprocs=1:ompthreads=N` variants belong in
their own cells, not this one.